# TrackNetV3 のファインチューニング（Google Colab）

PC に GPU がない場合に、Colab の GPU で TrackNetV3 をファインチューニングするためのノートブック。

1. ランタイム → ランタイムのタイプを変更 → **GPU** を選ぶ
2. PC で `smash-speed export-tracknet --index data/index.csv --out data/tracknet_dataset` を実行し、
   `tracknet_dataset` フォルダを Google Drive の `MyDrive/smash-speed/` にアップロードしておく
3. TrackNetV3 の README にある Google Drive のリンクから学習済み重み（`TrackNet_best.pt`, `InpaintNet_best.pt`）を取得し、
   `MyDrive/smash-speed/ckpts/` に置いておく

TrackNetV3: https://github.com/qaz812345/TrackNetV3 （MIT License。学習データセットのライセンスは別途確認すること）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/smash-speed'

In [ ]:
!git clone https://github.com/qaz812345/TrackNetV3.git /content/TrackNetV3
%cd /content/TrackNetV3
!pip install -q -r requirements.txt
import torch; print(torch.__version__, torch.cuda.is_available())

## データと学習済み重みの配置

`dataset.py` の `data_dir` が `data` を指していることを確認する（違う場合は書き換える）。

In [ ]:
!rm -rf data && cp -r "$BASE/tracknet_dataset" data
!mkdir -p ckpts && cp "$BASE/ckpts/"*.pt ckpts/
!grep -n "data_dir" dataset.py | head
!ls data

In [ ]:
# フレーム画像・検証データ・背景画像の作成
!python preprocess.py

## 学習済み重みから再開する形でファインチューニング

TrackNetV3 の `train.py` は `--resume_training` で `<save_dir>/TrackNet_cur.pt` を読み込む。
学習済みの重みをこの名前でコピーし、総エポック数を（学習済みのエポック + 追加分）にして実行する。

In [ ]:
import torch, os, json
SAVE = 'exp_finetune'
EXTRA_EPOCHS = 10
os.makedirs(SAVE, exist_ok=True)
ckpt = torch.load('ckpts/TrackNet_best.pt', map_location='cpu')
print('学習済みの設定:', ckpt['param_dict'])
start = int(ckpt.get('epoch', 0))
ckpt['max_val_acc'] = 0.0  # 新しいデータで最良を保存し直す
torch.save(ckpt, f'{SAVE}/TrackNet_cur.pt')
TOTAL = start + 1 + EXTRA_EPOCHS
p = ckpt['param_dict']
print('総エポック数:', TOTAL)

In [ ]:
!python train.py --model_name TrackNet --seq_len {p['seq_len']} --epochs {TOTAL} --batch_size 10     --learning_rate 1e-4 --bg_mode {p['bg_mode'] or "''"} --alpha 0.5 --tolerance 4     --save_dir {SAVE} --resume_training --verbose 2>&1 | tee {SAVE}/finetune_log.txt

## 評価と保存

テストデータでの精度を確認し、重みとログを Drive に保存する。PC では
`configs/default.yaml` の `detection.tracknet.tracknet_ckpt` にこの重みを指定して `smash-speed detect --detector tracknet` を使う。

In [ ]:
!python test.py --tracknet_file {SAVE}/TrackNet_best.pt --save_dir {SAVE}/eval 2>&1 | tail -20
!mkdir -p "$BASE/finetuned" && cp {SAVE}/TrackNet_best.pt {SAVE}/finetune_log.txt "$BASE/finetuned/"